<a href="https://colab.research.google.com/github/tellanad/AI-For-Beginners/blob/main/10_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import re

# Step 1: Our very first training examples
examples = [
    ("show all circuits",
     "SELECT circuit_id FROM circuits LIMIT 100;"),

    ("show failed circuits",
     "SELECT circuit_id FROM circuits WHERE status = 'failed' LIMIT 100;"),

    ("count active circuits",
     "SELECT COUNT(*) FROM circuits WHERE status = 'active';")
]

# Step 2: Split English into words
def tokenize(text):
    return re.findall(r"[a-z_]+", text.lower())

# Step 3: Create a vocabulary
special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]

words = sorted({
    word
    for question, sql in examples
    for word in tokenize(question)
})

vocab = special_tokens + words

# Step 4: Assign numerical IDs
word_to_id = {word: i for i, word in enumerate(vocab)}

# Step 5: Convert a sentence into IDs
question = "show failed circuits"

tokens = tokenize(question)
ids = (
    [word_to_id["<bos>"]]
    + [word_to_id.get(t, word_to_id["<unk>"]) for t in tokens]
    + [word_to_id["<eos>"]]
)

print("Vocabulary:", word_to_id)
print("Tokens:", tokens)
print("Token IDs:", ids)

In [ ]:

events = [
    {"agent_id": "billing", "status": "success", "latency_ms": 120},
    {"agent_id": "orders", "status": "failed", "latency_ms": 300},
    {"agent_id": "billing", "status": "failed", "latency_ms": 180},
    {"agent_id": "billing", "status": "success", "latency_ms": 100},
    {"agent_id": "orders", "status": "success", "latency_ms": 200}
]


In [ ]:
import json
from typing import List, Dict, Any

def aggregate_agent_events(events: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregates events to compute total, success, failure, and average latency per agent.
    Optimized to run in O(N) time with O(1) extra space complexity per unique agent
    by avoiding storing individual latencies and statuses in lists.
    """
    # Use integer and float accumulators for optimal space efficiency
    stats = {}

    for event in events:
        agent = event["agent_id"]
        status = event["status"]
        latency = event["latency_ms"]

        if agent not in stats:
            stats[agent] = {
                "total": 0,
                "success": 0,
                "failure": 0,
                "total_latency": 0.0
            }

        stats[agent]["total"] += 1
        if status == "success":
            stats[agent]["success"] += 1
        elif status == "failed":
            stats[agent]["failure"] += 1

        stats[agent]["total_latency"] += latency

    # Generate the formatted response structure
    return {
        agent: {
            "total": data["total"],
            "success": data["success"],
            "failure": data["failure"],
            "avg_latency_ms": round(data["total_latency"] / data["total"], 2) if data["total"] > 0 else 0.0
        }
        for agent, data in stats.items()
    }

# --- Test Cases ---
def run_tests():
    test_events = [
        {"agent_id": "billing", "status": "success", "latency_ms": 120},
        {"agent_id": "orders", "status": "failed", "latency_ms": 300},
        {"agent_id": "billing", "status": "failed", "latency_ms": 180},
        {"agent_id": "billing", "status": "success", "latency_ms": 100},
        {"agent_id": "orders", "status": "success", "latency_ms": 200}
    ]

    result = aggregate_agent_events(test_events)

    # Assertions for validation
    assert result["billing"]["total"] == 3
    assert result["billing"]["success"] == 2
    assert result["billing"]["failure"] == 1
    assert result["billing"]["avg_latency_ms"] == 133.33

    assert result["orders"]["total"] == 2
    assert result["orders"]["success"] == 1
    assert result["orders"]["failure"] == 1
    assert result["orders"]["avg_latency_ms"] == 250.0

    print("All tests passed successfully!")
    return result

# Run tests and output formatted JSON
final_output = run_tests()
print("\nOutput result:")
print(json.dumps(final_output, indent=4))

{
    "billing": {
        "total": 3,
        "success": 2,
        "failure": 1,
        "avg_latency_ms": 133.33
    },
    "orders": {
        "total": 2,
        "success": 1,
        "failure": 1,
        "avg_latency_ms": 250.0
    }
}


In [ ]:
import json
from collections import defaultdict

# Automatically initializes missing keys with an empty list
grouped_data = defaultdict(list)
for event in events:
    # No "if agent not in grouped_data:" check needed!
    grouped_data[event["agent_id"]].append(event["latency_ms"])

# To print it cleanly as a standard JSON-like dictionary:
print(json.dumps(dict(grouped_data), indent=4))

{
    "billing": [
        120,
        180,
        100
    ],
    "orders": [
        300,
        200
    ]
}


In [ ]:
import json
from collections import defaultdict
from typing import List, Dict, Any

def process_agent_metrics(events: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregates metrics per agent in O(N) time and O(1) extra space per agent.
    Avoids storing list arrays of latencies/statuses to maximize efficiency.
    """
    # Initialize with default structure representing counts and running sums
    grouped_data = defaultdict(lambda: {"total": 0, "success": 0, "failure": 0, "total_latency": 0.0})

    for event in events:
        agent = event["agent_id"]
        status = event["status"]
        latency = event["latency_ms"]

        metrics = grouped_data[agent]
        metrics["total"] += 1
        metrics["total_latency"] += latency

        if status == "success":
            metrics["success"] += 1
        elif status == "failed":
            metrics["failure"] += 1

    # Map into the final formatted layout
    return {
        agent: {
            "total": data["total"],
            "success": data["success"],
            "failure": data["failure"],
            "avg_latency_ms": round(data["total_latency"] / data["total"], 2) if data["total"] > 0 else 0.0
        }
        for agent, data in grouped_data.items()
    }

# --- Unit Tests ---
def run_validation_tests():
    test_cases = [
        {"agent_id": "billing", "status": "success", "latency_ms": 120},
        {"agent_id": "orders", "status": "failed", "latency_ms": 300},
        {"agent_id": "billing", "status": "failed", "latency_ms": 180},
        {"agent_id": "billing", "status": "success", "latency_ms": 100},
        {"agent_id": "orders", "status": "success", "latency_ms": 200}
    ]

    result = process_agent_metrics(test_cases)

    # Verification assertions
    assert result["billing"]["total"] == 3
    assert result["billing"]["success"] == 2
    assert result["billing"]["failure"] == 1
    assert result["billing"]["avg_latency_ms"] == 133.33

    assert result["orders"]["total"] == 2
    assert result["orders"]["success"] == 1
    assert result["orders"]["failure"] == 1
    assert result["orders"]["avg_latency_ms"] == 250.0

    print("All test assertions passed successfully!")
    return result

# Run the tests and print output
final_output = run_validation_tests()
print("\nFinal Output Result:")
print(json.dumps(final_output, indent=4))

All test assertions passed successfully!

Final Output Result:
{
    "billing": {
        "total": 3,
        "success": 2,
        "failure": 1,
        "avg_latency_ms": 133.33
    },
    "orders": {
        "total": 2,
        "success": 1,
        "failure": 1,
        "avg_latency_ms": 250.0
    }
}


In [2]:


import time

short_prompt = "What is a database index?"

long_prompt = (
    "A database index helps locate rows efficiently. " * 50
    + "What is a database index?"
)

for label, prompt in [
    ("SHORT", short_prompt),
    ("LONG", long_prompt),
]:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    # Repeat once to reduce first-run overhead
    for repeat in range(2):
        start = time.perf_counter()

        with torch.inference_mode():
            model.generate(
                **inputs,
                max_new_tokens=1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        elapsed = time.perf_counter() - start

    print(f"{label}:")
    print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
    print(f"  Generation time: {elapsed:.3f}s")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

SHORT:
  Input tokens: 35
  Generation time: 3.146s
LONG:
  Input tokens: 435
  Generation time: 14.126s
